In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation, BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "pretraining_results_with_metadata.csv"

df_all = pd.read_csv(results_file)

In [ ]:
df_all

In [ ]:
METRIC = "cka_ve_test"

In [ ]:
from matplotlib.colors import LogNorm


nrows, ncols = 1, 1
subfigsize = (10, 7)
subfigsize = (10, 5)
zoom = 0.75
figsize = (subfigsize[0] * zoom * ncols, subfigsize[1] * zoom * nrows)


fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=figsize, 
    # sharey=True,
    dpi = 300
)
ax = axes

data = df_all.copy()
data = data[data.model_id.apply(lambda x: 'flex' not in x.lower())]
data = data[(data.is_adversarially_finetuned != True) & (data.is_selfsupervised_learning != True)]


data = apply_hiearchical_aggregation(
    df=data,
    groupby_cols=GENERIC_GROUPING_COLUMNS,
    agg_cols=METRICS_COMPACT,
    agg_func='mean'
)


color_palette = sns.color_palette("viridis", as_cmap=True, n_colors=10)


vmin = data["backbone_n_params"].min()
vmax = data["backbone_n_params"].max()
norm = LogNorm(vmin=vmin, vmax=vmax)
cmap = plt.cm.viridis  # use mpl colormap directly
cmap = plt.cm.magma

marker_map = {
    "spvvs": "o",
    "timm": "X",
}
# ---- marker size scaling (area in pt^2) ----
s_min, s_max = 25, 220   # tune these

def size_from_params(p):
    # map params -> [0,1] using your LogNorm, then to marker area
    t = float(norm(p))   # in [0,1]
    return s_min + t * (s_max - s_min)

for arch in data.backbone_arch.unique():
    df_arch = data[data.backbone_arch == arch].copy()

    # draw separate curves per (source, params) so color + marker are consistent
    for (source, p), g in df_arch.groupby(["backbone_source", "backbone_n_params"]):
        g = g.sort_values("pretraining_total_flops_estimate")
        color = cmap(norm(p))
        marker = marker_map.get(source, "o")

        # line (no markers)
        ax.plot(
            g["pretraining_total_flops_estimate"].values,
            g[METRIC].values,
            linestyle="-",
            linewidth=1.5,
            color=color,
            alpha=0.9,
            zorder=2,
        )

        # markers with size scaled by params
        ax.scatter(
            g["pretraining_total_flops_estimate"].values,
            g[METRIC].values,
            s=size_from_params(p),
            marker=marker,
            c=[color],
            edgecolors="none",
            alpha=0.9,
            zorder=3,
        )
ax.set_xscale('log')
ax.set_xlabel('Pretraining FLOPs', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Alignment (S)', fontsize=12, fontweight='bold')
ax.set_title('Scaling Pretraining Compute per Model', fontsize=16, fontweight='bold')

# plt.colorbar(label='Number of Parameters')

sm = plt.cm.ScalarMappable(
    cmap= cmap, 
    norm=norm
)
sm.set_clim(data['backbone_n_params'].min(), data['backbone_n_params'].max())
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Number of Parameters', rotation=270, labelpad=15, fontsize=16, fontweight='bold')



from matplotlib.lines import Line2D

guide_params = np.array([5e6, 1e7, 3e7, 1e8, 3e8, 8e8])
handles = []
for p in guide_params:
    handles.append(
        Line2D(
            [0], [0],
            marker="o",
            linestyle="-",
            color=cmap(norm(p)),
            markersize=6 + 4 * (norm(p) - norm(vmin)) / (norm(vmax) - norm(vmin)),
            label=f"{p/1e6:.0f}M params",
        )
    )

size_legend = ax.legend(
    handles=handles,
    title="Model size",
    frameon=True,
    loc="center right",   # choose a spot that doesn't collide
)

# IMPORTANT: keep this legend when adding another
ax.add_artist(size_legend)

source_handles = [
    Line2D(
        [0], [0],
        marker=marker,
        linestyle="None",
        color="black",
        label=source,
        markersize=8,
    )
    for source, marker in marker_map.items()
    if source in data["backbone_source"].unique()
]

source_legend = ax.legend(
    handles=source_handles,
    title="Model source",
    frameon=True,
    loc="lower right",
)

ax.grid(True, which="both", ls="--", lw=0.5)

ax.spines[['right', 'top']].set_visible(False)

# ax.set_ylim(0, 1.0)
cbar.remove()
plt.tight_layout()

figures_dir = save_dir
fig_name = f'metrics-pretraining-{METRIC}-compute-per_model'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)